In [ ]:
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
PROJECT_ROOT = Path( "/Volumes/code/MyProjects/ML/University Handbook Assistant From Scratch")
raw_dir= PROJECT_ROOT / "data"/"raw"
processed_dir = PROJECT_ROOT / "data" / "processed"

def get_pdf_text(file_path):
    document =[]
    files=sorted(
        path for path in file_path.iterdir() if path.is_file() and path.suffix.lower() in {".pdf", ".txt"}
        )

    for path in files:
        if path.is_file() and path.suffix.lower() == ".pdf" :
            pdfReader=PdfReader(path)
            for page_number, page in enumerate(pdfReader.pages):
                text=page.extract_text() or ""
                document.append({
                    "file_name": path.name,
                    "page_number": page_number + 1,
                    "text": text
                })

        if path.is_file() and path.suffix.lower() == ".txt":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                document.append({
                    "file_name": path.name,
                    "page_number": 1,
                    "text": text
                })
    
    return document
                

document = get_pdf_text(raw_dir)   



print(f"Found {document[0]}")

In [ ]:
def do_chunking(documents, chunk_size=1200, overlap=200):
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")

    chunks = []

    for doc in documents:
        text = doc["text"].strip()
        start = 0
        chunk_number = 1

        while start < len(text):
            end = min(start + chunk_size, len(text))

            chunks.append({
                "filename": doc["file_name"],
                "chunk_id": (
                    f"{doc['file_name']}::page-{doc['page_number']}"
                    f"::chunk-{chunk_number}"
                ),
                "page": doc["page_number"],
                "text": text[start:end],
            })

            if end == len(text):
                break

            start = end - overlap
            chunk_number += 1

    return chunks

document_chunks = do_chunking(document, chunk_size=1200, overlap=200)

In [ ]:
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

In [ ]:
def do_embedding(chunks, model):
    if not chunks:
        raise ValueError("No chunks to embed.")

    passages = [
        f"passage: {chunk['text']}"
        for chunk in chunks
    ]

    embeddings = model.encode(
        passages,
        batch_size=16,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    for chunk, embedding in zip(chunks, embeddings):
        chunk["embedding"] = embedding

    return embeddings

embeddings = do_embedding(document_chunks, embedding_model)

In [ ]:
import json
import numpy as np


def save_chunks_embeddings(chunks, output_dir, embedding_model):
    output_dir.mkdir(parents=True, exist_ok=True)
    embeddings = do_embedding(chunks, embedding_model)
    
    # Keep vectors separate from the JSON metadata.
    saved_chunks = [
        {key: value for key, value in chunk.items() if key != "embedding"}
        for chunk in chunks
    ]

    if len(saved_chunks) != len(embeddings):
        raise ValueError("Chunk and embedding counts do not match.")

    (processed_dir / "chunks.json").write_text(
        json.dumps(saved_chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    
    np.save(processed_dir / "embeddings.npy", embeddings)

    config = {
        "embedding_model": "intfloat/multilingual-e5-base",
        "normalized": True,
        "chunk_count": len(saved_chunks),
        "embedding_dimension": int(embeddings.shape[1]),
    }

    (processed_dir / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    print("Saved to:", processed_dir)

save_chunks_embeddings(document_chunks, processed_dir, embedding_model)

In [ ]:
# long_page = next(
#     (
#         page for page in document
#         if len(page["text"].strip()) > 1200
#     ),
#     None,
# )

# if long_page is None:
#     print("No page exceeds 1,200 characters.")
# else:
#     print("File:", long_page["file_name"])
#     print("Page:", long_page["page_number"])
#     print("Characters:", len(long_page["text"]))
#     print("\nText with visible line breaks:")
#     print(repr(long_page["text"]))

In [ ]:
# page_text = long_page["text"].strip()
# model_input = f"passage: {page_text}"

# token_count = len(
#     embedding_model.tokenizer.encode(
#         model_input,
#         truncation=False,
#     )
# )

# token_limit = embedding_model.max_seq_length

# print("Page tokens:", token_count)
# print("Model limit:", token_limit)

# if token_count <= token_limit:
#     print("Keep this page as one chunk.")
# else:
#     print("Split this page into smaller chunks.")

In [ ]:
# pages_to_split = []
# whole_page_count = 0
# empty_page_count = 0

# for page in document:
#     text = (page["text"] or "").strip()

#     if not text:
#         empty_page_count += 1
#         continue

#     token_count = len(
#         embedding_model.tokenizer.encode(
#             f"passage: {text}",
#             truncation=False,
#         )
#     )

#     if token_count <= embedding_model.max_seq_length:
#         whole_page_count += 1
#     else:
#         pages_to_split.append({
#             "file_name": page["file_name"],
#             "page_number": page["page_number"],
#             "tokens": token_count,
#         })

# print("Keep whole:", whole_page_count)
# print("Need splitting:", len(pages_to_split))
# print("Empty:", empty_page_count)

# for page in pages_to_split:
#     print(page)

In [ ]:
def do_chunking(documents):
    chunks = []

    for doc in documents:
        text = (doc["text"] or "").strip()

        if not text:
            continue

        chunks.append({
            "filename": doc["file_name"],
            "chunk_id": (
                f"{doc['file_name']}::page-{doc['page_number']}::chunk-1"
            ),
            "page": doc["page_number"],
            "text": text,
        })

    return chunks

In [ ]:
document_chunks = do_chunking(document)
save_chunks_embeddings(document_chunks, processed_dir, embedding_model)